# Group Name: AISE 3350A 20

HI

# Install Dependencies (READ THIS)
- When choosing a kernal, select "+ CREATE PYTHON ENVIORNMENT" --> "VENV" --> Python 3.10.18
- It is important to use a python 3.9 - python 3.12 enviornment becuase MediaPipe Hands only supports python 3.9 - python 3.12

In [9]:
!pip install pandas
!pip install
!pip install scikit-learn
!pip install opencv-python
!pip install mediapipe

ERROR: You must give at least one requirement to install (see "pip help install")
  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl (5.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
  Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into acc

# Split data, Train RFC, Evaluate Model
- The model is trained on **rps_landmarks.csv** which was created within **feature_extraction.ipynb**

**rps_landmarks.csv**: Dataset containing samples with labels 'rock', 'paper', or 'scissors' and their corresponding hand landmark coordinates. The dataset has the shape (2040, 43) with 2040 samples (≈33% per label), 1 column for labels, 21 columns for X coordinates, and 21 columns for Y coordinates.

**feature_extraction.ipynb**: The features within the training dataset were extracted from a kaggle dataset containing photos of rock, paper, and scissor hand gestures. These images were fed into OpenCV's MediaPipe Hand Detection and Landmark Estimation and the coordinates were saved in the dataset.

Link to kaggle dataset: https://www.kaggle.com/datasets/drgfreeman/rockpaperscissors

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

df = pd.read_csv('rps_landmarks.csv')

X = df.drop('label', axis=1).values #labels are our target
y = df['label'].values #.values converts to numpy array, which is what we expect for our input

#split 80/20 train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#initialize rfc model
model = RandomForestClassifier(
    n_estimators=100, #
    random_state=42, 
    n_jobs=-1,  #use all cores
    max_depth=10 
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, labels=model.classes_))

              precision    recall  f1-score   support

       paper       0.99      1.00      0.99       139
        rock       0.99      1.00      1.00       135
    scissors       1.00      0.98      0.99       134

    accuracy                           0.99       408
   macro avg       0.99      0.99      0.99       408
weighted avg       0.99      0.99      0.99       408

[[139   0   0]
 [  0 135   0]
 [  2   1 131]]


In [16]:
df.shape

(2040, 43)

# Save the Trained RFC Model as .pkl
- Model is saved in the same location as this notebook

In [11]:
import pickle
import os

# Define the file path for the saved model
MODEL_FILENAME = 'rps_classifier_V1.pkl'

try:
    with open(MODEL_FILENAME, 'wb') as file:
        # Dump the trained model object into the file
        pickle.dump(model, file)
    print(f"Saved to {MODEL_FILENAME}")

except Exception as e:
    print(f"Error saving the model: {e}")

Saved to rps_classifier_V1.pkl


# Helper Functions to Extract Hand Landmarks
- INPUT a sample photo of 4 hands --> OUTPUT the coordinates of the landmarkers on each hand

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

def extract_hand_landmark_features(hand_landmarks):  
    landmark_features = []

    #Use the wrist (landmark 0) for normalization reference
    base_x, base_y = hand_landmarks[0].x, hand_landmarks[0].y

    for landmark in hand_landmarks:
        # NORMALIZATION USED DURING TRAINING. POSITION RELATIVE TO WRIST
        landmark_features.append(landmark.x - base_x)
        landmark_features.append(landmark.y - base_y)

    #NORMALIZE AGAIN. USE DISTANCE FROM WRIST TO TIP OF MIDDLE FINGER AS REFERENCE
    X9_norm=landmark_features[18]
    Y9_norm=landmark_features[19]

    #reference length calculations
    L_ref = ((X9_norm)**2 + (Y9_norm)**2)**0.5

    final_features = []
    for coord in landmark_features:
        final_features.append(coord / L_ref)

    return final_features #list of 42 normalized features (x and y for each of 21 landmarks)

def plot_hand_landmarks_2d(hand_landmarks, handNum):
    x_coords = [landmark.x for landmark in hand_landmarks]
    y_coords = [landmark.y for landmark in hand_landmarks]

    #Create a scatter plot
    plt.figure(figsize=(6, 6))
    plt.scatter(x_coords, y_coords, c='blue', label='Landmarks')

    #Connect hand landmarks with lines
    for connection in mp_hands.HAND_CONNECTIONS:
        start_idx, end_idx = connection
        plt.plot(
            [x_coords[start_idx], x_coords[end_idx]],
            [y_coords[start_idx], y_coords[end_idx]],
            'r-'
        )

    plt.gca().invert_yaxis()  # Invert Y-axis to match image coordinates
    plt.title(f"Hand {handNum} Landmarks (2D)")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.show()

#function used to grab hand landmarks from input images
def get_landmarks(IMAGE_FILES):
    normalized_features_list = []
    wrist_coords=[]
    handNum = 1

    with mp_hands.Hands(
        static_image_mode=True, #media pipe setting for static images
        max_num_hands=4,
        min_detection_confidence=0.1) as hands:
        for file in IMAGE_FILES:
            image = cv2.imread(file)
            results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)) #process the image to find hands. Covert to numPy array cuz media pipe uses that format.

            if not results.multi_hand_landmarks: #if no hands detected, skip to next image
                continue
            
            #Iterate over each detected hand
            for hand_landmarks in results.multi_hand_landmarks: #hand_landmarks is a single hand found in the image. multi_hand_landmarks is a list of all hands found. iterate
                # print ("-"*40)
                # print(f"Hand {handNum}: Wrist coordinates NOT NORMALIZED: (",
                    #   f"{hand_landmarks.landmark[0].x}, " 
                    #   f"{hand_landmarks.landmark[0].y})")
                # print ("-"*40)
                wrist_coords.append( (hand_landmarks.landmark[0].x, hand_landmarks.landmark[0].y) )
                
                # Extract and print normalized features for the current hand
                normalized_features = extract_hand_landmark_features(hand_landmarks.landmark) #.landmark is a list of 21 landmarks for the hand
                normalized_features_list.append(normalized_features)
                # print ("-"*40)
                # print(f"Hand {handNum}: {normalized_features}")
                # print ("-"*40)

                # Plot the landmarks for the current hand (COMMENT OUT IF U DONT WANNA SEE HAND GRAPHS)
                # plot_hand_landmarks_2d(hand_landmarks.landmark, handNum)

                handNum += 1

    return normalized_features_list, wrist_coords

# Helper Functions to Identify Player Hands
- determains which hands are YOURS and which hands are the OPPONENTS

In [ ]:
def identify_hands(wrist_coords, hand_labels):

    #grabs y coordinate from wrist
    def sort_by_y(hand):
        return hand[0][1]

    #zip combines wrist coord and labels into tuples e.g.(0.5, 0.8), "rock")...
    #key = sort_by_y means sort by the y coordinate of the wrist
    #hands_with_labels is now sorted by y coordinate of wrist from lowest to highest
    hands_with_labels = sorted(zip(wrist_coords, hand_labels), key=sort_by_y) 
    my_hands = hands_with_labels[2:] #we know highest y cords are mine
    opponent_hands = hands_with_labels[:2] #lowest y cords are opponent's

    #sort my hands ascending to determine left and right
    my_hands = sorted(my_hands, key=lambda hand: hand[0][0])
    my_left = my_hands[0][1]
    my_right = my_hands[1][1]

    #sort opponent hands ascending to determine left and right
    opponent_hands = sorted(opponent_hands, key=lambda hand: hand[0][0])
    opponent_left = opponent_hands[0][1]
    opponent_right = opponent_hands[1][1]

    return { #output as dictionary
        'my_left': my_left,
        'my_right': my_right,
        'opponent_left': opponent_left,
        'opponent_right': opponent_right
    }

# MAIN
- adjust IMAGE_FILES to the path of your desired photo

In [17]:
import pickle
from sklearn.ensemble import RandomForestClassifier
import numpy as np

################################################################################
IMAGE_FILES = ['sample_photos/TEST1.jpg'] #CHANGE THIS TO TEST DIFFERENT IMAGES
################################################################################

landmarks, wrist_coords = get_landmarks(IMAGE_FILES)

MODEL_FILENAME = 'rps_classifier_V1.pkl'
store_predictions = []

try:
    with open(MODEL_FILENAME, 'rb') as file: #rb means read binary
        rps_classifier = pickle.load(file)
    
    for i in range(len(landmarks)):
        prediction = rps_classifier.predict(np.array(landmarks[i]).reshape(1,-1)) #landmarks is a list of lists. Convert to numpy array for prediction
        # print(f"Hand {i+1} Predicted Gesture: {prediction[0].upper()}") #prediction is an array, so get first element
        store_predictions.append(prediction[0])

except FileNotFoundError: #throw error if model file not found
    print(f"Error: Model file '{MODEL_FILENAME}' not found.")
except Exception as e: #throw error for other issues
    print(f"Error during prediction: {e}")

hands = identify_hands(wrist_coords, store_predictions)
print (hands)



{'my_left': 'paper', 'my_right': 'scissors', 'opponent_left': 'scissors', 'opponent_right': 'rock'}


I0000 00:00:1764901207.754388  248718 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
W0000 00:00:1764901207.787541  268019 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764901207.798323  268020 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
